# Winger Analysis System Documentation

## Overview
The Winger Analysis System is a sophisticated tool designed to analyze and evaluate the performance of football/soccer wingers using StatsBomb event data. The system processes match events to generate comprehensive performance metrics across multiple dimensions of winger play.

## Core Components

### 1. CompetitionConfig Class
- **Purpose**: Manages competition-specific configurations and thresholds
- **Key Features**:
  - Initializes competition and season parameters
  - Calculates minimum minutes threshold for player inclusion based on competition size
  - Handles varying competition sizes with adaptive thresholds

### 2. Performance Metrics

#### 2.1 Finishing Quality (calculate_finishing_quality)
- Evaluates shooting efficiency and goal-scoring ability
- Components:
  - Basic conversion rate (40% weight)
  - Expected goals (xG) performance (40% weight)
  - Shot location quality (20% weight)
- Scale: 0-1, normalized to 4-10 in final output

#### 2.2 Chance Creation (calculate_chance_creation)
- Measures creative output and assistive play
- Components:
  - Key pass rate (40% weight)
  - Assist rate (30% weight)
  - Cross completion rate (30% weight)
- Minimum 15 passes required for evaluation

#### 2.3 Ball Progression (calculate_ball_progression)
- Analyzes ability to move the ball forward
- Components:
  - Progressive passes (50% weight)
  - Progressive carries (50% weight)
- Calculates progression based on pitch coordinates

#### 2.4 One-v-One Ability (calculate_one_v_one)
- Evaluates dribbling and take-on success
- Components:
  - Dribble success rate (70% weight)
  - Progressive value of dribbles (30% weight)
- Minimum 10 dribble attempts required

#### 2.5 Off-Ball Movement (calculate_off_ball_movement)
- Assesses movement and positioning without the ball
- Components:
  - Final third receives (60% weight)
  - Box entry rate (40% weight)
- Minimum 15 ball receives required

#### 2.6 Defensive Contribution (calculate_defensive_contribution)
- Measures defensive work rate and effectiveness
- Components:
  - Pressure success rate (70% weight)
  - Ball recovery rate (30% weight)
- Minimum 15 pressure actions required

## Analysis Functions

### 1. analyze_winger_performance
- **Purpose**: Main analysis function for single competition/season
- **Process**:
  1. Identifies players in winger positions
  2. Calculates all performance metrics
  3. Normalizes scores (0-1 scale to 4-10 scale)
  4. Computes composite score
- **Output**: DataFrame with player metrics and rankings

### 2. analyze_multiple_leagues
- **Purpose**: Aggregates analysis across multiple competitions
- **Features**:
  - Handles duplicate players across leagues
  - Prioritizes entries with most minutes played
  - Exports results to CSV
- **Output**: Combined DataFrame with unique players

## Data Processing

### Score Normalization
- All raw metrics are clipped to 0-1 range
- Final scores are scaled to 4-10 range for better interpretation
- Weighted averages used for player metrics across multiple matches

### Quality Controls
- Minimum sample size requirements for each metric
- Default scores (0.5) assigned when sample size is insufficient
- Minutes played and matches tracked for context

## Usage

### Required Dependencies
- pandas
- numpy
- statsbombpy
- scipy
- typing

### Example Usage
```python
leagues = [
    {"competition_id": 9, "season_ids": [281]},
    {"competition_id": 43, "season_ids": [106]},
    # Add more leagues as needed
]

result_df = analyze_multiple_leagues(leagues)
```

### Output Format
The system generates a CSV file containing:
- Player identification (name, team)
- Competition and season information
- Match participation stats
- Performance metrics (all on 4-10 scale)
- Composite score
- Minutes played and match count

## Error Handling
- Robust error handling at multiple levels
- Continues analysis even if individual matches/players fail
- Returns empty DataFrame if critical errors occur

## Limitations
- Requires StatsBomb event data access
- Performance dependent on data quality
- Minimum playing time requirements may exclude some players
- Metrics based on available event data only

In [ ]:
import pandas as pd
import numpy as np
from typing import List, Dict, Union, Tuple
from statsbombpy import sb
from collections import defaultdict
from scipy import stats
import warnings
import glob
import os

warnings.simplefilter("ignore")

class CompetitionConfig:
    def __init__(self, competition_id: int, season_id: int):
        """
        Initialize competition configuration
        """
        self.competition_id = competition_id
        self.season_id = season_id
        matches = sb.matches(competition_id=self.competition_id, season_id=self.season_id)
        self.total_matches = len(matches)
    
    def get_minimum_minutes(self) -> int:
        """
        Calculate minimum minutes threshold for analysis
        """
        total_possible_minutes = self.total_matches * 90
        if self.total_matches <= 7:
            return 180
        elif self.total_matches <= 15:
            return total_possible_minutes * 0.15
        else:
            return total_possible_minutes * 0.10

def calculate_finishing_quality(events: pd.DataFrame, player: str, team: str) -> float:
    """
    Calculate finishing quality with contextual adjustments
    Returns a score between 0-1
    """
    player_events = events[events['player'] == player]
    shots = player_events[player_events['type'] == 'Shot']
    
    if len(shots) < 10:
        return 0.5
    
    # Basic conversion rate
    goals = shots[shots['shot_outcome'] == 'Goal']
    conversion_rate = len(goals) / len(shots) if len(shots) > 0 else 0
    
    # xG analysis
    xg_performance = shots['shot_statsbomb_xg'].sum() / len(shots) if 'shot_statsbomb_xg' in shots.columns else 0.5
    
    # Shot location quality
    def get_shot_location_value(location: List[float]) -> float:
        if not isinstance(location, list) or len(location) != 2:
            return 0
        x, y = location
        central_bonus = 1 - (abs(y - 40) / 40)
        distance_factor = x / 120
        return central_bonus * distance_factor
    
    location_quality = shots['location'].apply(get_shot_location_value).mean()
    
    # Combine metrics
    return (conversion_rate * 0.4 + xg_performance * 0.4 + location_quality * 0.2)


def calculate_chance_creation(events: pd.DataFrame, player: str, team: str) -> float:
    """
    Calculate chance creation ability with key passes and assists
    Returns a score between 0-1
    """
    try:
        player_events = events[events['player'] == player]
        passes = player_events[player_events['type'] == 'Pass']
        
        if len(passes) < 15:
            return 0.5
        
        # Safely check for key passes and assists
        key_passes = passes.get('pass_shot_assist', pd.Series([False] * len(passes))).fillna(False)
        assists = passes.get('pass_goal_assist', pd.Series([False] * len(passes))).fillna(False)
        
        key_pass_rate = sum(key_passes) / len(passes)
        assist_rate = sum(assists) / len(passes)
        
        # Cross completion
        crosses = passes[passes['pass_type'] == 'Cross']
        cross_completion = (
            len(crosses[crosses['pass_outcome'].isna()]) / len(crosses)
            if len(crosses) > 0 else 0
        )
        
        return (key_pass_rate * 0.4 + assist_rate * 0.3 + cross_completion * 0.3)
    except:
        return 0.5
def calculate_ball_progression(events: pd.DataFrame, player: str, team: str) -> float:
    """
    Calculate ball progression through carries and passes
    Returns a score between 0-1
    """
    player_events = events[events['player'] == player]
    
    # Progressive passes
    passes = player_events[player_events['type'] == 'Pass']
    def calculate_pass_progression(start_loc: List[float], end_loc: List[float]) -> float:
        if not (isinstance(start_loc, list) and isinstance(end_loc, list)):
            return 0
        start_x, _ = start_loc
        end_x, _ = end_loc
        return max(0, (end_x - start_x) / 30)
    
    pass_progression = passes.apply(
        lambda x: calculate_pass_progression(x['location'], x['pass_end_location']),
        axis=1
    ).mean() if len(passes) > 0 else 0
    
    # Carries
    carries = player_events[player_events['type'] == 'Carry']
    carry_progression = carries.apply(
        lambda x: calculate_pass_progression(x['location'], x['carry_end_location']),
        axis=1
    ).mean() if len(carries) > 0 else 0
    
    return (pass_progression * 0.5 + carry_progression * 0.5)

def calculate_one_v_one(events: pd.DataFrame, player: str, team: str) -> float:
    """
    Calculate one-v-one effectiveness through dribbling
    Returns a score between 0-1
    """
    player_events = events[events['player'] == player]
    dribbles = player_events[
        (player_events['type'] == 'Duel') & 
        (player_events['duel_type'] == 'Take On')
    ]
    
    if len(dribbles) < 10:
        return 0.5
    
    successful_dribbles = len(dribbles[dribbles['duel_outcome'] == 'Won'])
    dribble_success_rate = successful_dribbles / len(dribbles)
    
    # Progressive value of dribbles
    def calculate_dribble_progression(location: List[float], end_location: List[float]) -> float:
        if not (isinstance(location, list) and isinstance(end_location, list)):
            return 0
        start_x, _ = location
        end_x, _ = end_location
        return max(0, (end_x - start_x) / 15)
    
    progression_value = dribbles.apply(
        lambda x: calculate_dribble_progression(x['location'], x['duel_end_location']),
        axis=1
    ).mean()
    
    return (dribble_success_rate * 0.7 + progression_value * 0.3)

def calculate_off_ball_movement(events: pd.DataFrame, player: str, team: str) -> float:
    """
    Calculate effectiveness of off-ball movement
    Returns a score between 0-1
    """
    player_events = events[events['player'] == player]
    receives = player_events[player_events['type'] == 'Ball Receipt']
    
    if len(receives) < 15:
        return 0.5
    
    # Final third receives
    def is_final_third(location: List[float]) -> bool:
        return isinstance(location, list) and len(location) == 2 and location[0] >= 60
    
    final_third_rate = receives['location'].apply(is_final_third).mean()
    
    # Box entries
    def is_box_entry(location: List[float]) -> bool:
        if not isinstance(location, list) or len(location) != 2:
            return False
        x, y = location
        return x >= 75 and 15 <= y <= 65
    
    box_entry_rate = receives['location'].apply(is_box_entry).mean()
    
    return (final_third_rate * 0.6 + box_entry_rate * 0.4)

def calculate_defensive_contribution(events: pd.DataFrame, player: str, team: str) -> float:
    """
    Calculate defensive contribution through pressing and recoveries
    Returns a score between 0-1
    """
    player_events = events[events['player'] == player]
    
    # Pressing
    pressures = player_events[player_events['type'] == 'Pressure']
    if len(pressures) < 15:
        return 0.5
    
    # Pressure success (if team regains within 5 seconds)
    def pressure_successful(events: pd.DataFrame, pressure_idx: int) -> float:
        if pressure_idx + 3 >= len(events):
            return 0
        next_events = events.iloc[pressure_idx + 1:pressure_idx + 4]
        return 1.0 if any(event['team'] == team for _, event in next_events.iterrows()) else 0
    
    pressure_success_rate = sum(
        pressure_successful(events, i) for i in pressures.index
    ) / len(pressures)
    
    # Ball recoveries
    recoveries = len(player_events[player_events['type'] == 'Ball Recovery'])
    recovery_rate = recoveries / len(player_events)
    
    return (pressure_success_rate * 0.7 + recovery_rate * 0.3)

def analyze_winger_performance(competition_id: int, season_id: int) -> pd.DataFrame:
    """
    Analyze winger performance across multiple metrics
    Returns a DataFrame with normalized scores
    """
    try:
        config = CompetitionConfig(competition_id, season_id)
        matches = sb.matches(competition_id=competition_id, season_id=season_id)
        all_winger_stats = []
        
        for _, match in matches.iterrows():
            try:
                events = sb.events(match_id=match['match_id'])
                wingers = events[events['position'].isin([
                    'Right Wing', 'Left Wing', 'Right Winger', 'Left Winger'
                ])]['player'].unique()
                
                for winger in wingers:
                    try:
                        winger_events = events[events['player'] == winger]
                        if len(winger_events) == 0:
                            continue
                        
                        team = winger_events['team'].iloc[0]
                        
                        start_minute = winger_events['minute'].min()
                        end_minute = winger_events['minute'].max()
                        added_time = winger_events['second'].max() / 60 if end_minute >= 90 else 0
                        minutes = end_minute - start_minute + added_time
                        
                        metrics = {
                            'player': winger,
                            'team': team,
                            'match_id': match['match_id'],
                            'minutes_played': minutes,
                            'finishing_quality': calculate_finishing_quality(events, winger, team),
                            'chance_creation': calculate_chance_creation(events, winger, team),
                            'ball_progression': calculate_ball_progression(events, winger, team),
                            'one_v_one': calculate_one_v_one(events, winger, team),
                            'off_ball_movement': calculate_off_ball_movement(events, winger, team),
                            'defensive_contribution': calculate_defensive_contribution(events, winger, team)
                        }
                        
                        all_winger_stats.append(metrics)
                    except:
                        continue
            except:
                continue
        
        if not all_winger_stats:
            return pd.DataFrame()
        
        df = pd.DataFrame(all_winger_stats)
        metrics = [
            'finishing_quality', 'chance_creation', 'ball_progression',
            'one_v_one', 'off_ball_movement', 'defensive_contribution'
        ]
        
        def weighted_avg(x):
            return np.average(x, weights=df.loc[x.index, 'minutes_played'])
        
        grouped_df = df.groupby(['player', 'team']).agg({
            'minutes_played': 'sum',
            **{metric: weighted_avg for metric in metrics}
        }).reset_index()
        
        matches_played = df.groupby('player')['match_id'].nunique()
        grouped_df['matches_played'] = grouped_df['player'].map(matches_played)
        grouped_df['full_match_equivalent'] = (grouped_df['minutes_played'] / 90).round(1)
        
        for metric in metrics:
            grouped_df[metric] = grouped_df[metric].clip(0, 1)
            grouped_df[metric] = (grouped_df[metric] * 6) + 4
        
        grouped_df['composite_score'] = grouped_df[metrics].mean(axis=1)
        grouped_df = grouped_df.sort_values('composite_score', ascending=False)
        round_cols = metrics + ['composite_score', 'minutes_played']
        grouped_df[round_cols] = grouped_df[round_cols].round(2)
        
        return grouped_df.set_index('player')
    except:
        return pd.DataFrame()

import pandas as pd
import numpy as np
from statsbombpy import sb
from typing import List, Dict

def analyze_multiple_leagues(leagues: List[Dict]) -> pd.DataFrame:
    """
    Analyze winger performance across multiple leagues and seasons
    Handles duplicate players by keeping entry with most minutes played
    """
    all_analyses = []
    
    for league in leagues:
        competition_id = league["competition_id"]
        for season_id in league["season_ids"]:
            try:
                print(f"Analyzing competition {competition_id}, season {season_id}...")
                analysis = analyze_winger_performance(competition_id, season_id)
                if not analysis.empty:
                    analysis['competition_id'] = competition_id
                    analysis['season_id'] = season_id
                    all_analyses.append(analysis)
            except:
                continue
    
    if not all_analyses:
        return pd.DataFrame()
    
    combined_df = pd.concat(all_analyses, axis=0)
    combined_df = combined_df.reset_index()
    combined_df = combined_df.sort_values('minutes_played', ascending=False)
    combined_df = combined_df.drop_duplicates(subset=['player'], keep='first')
    combined_df = combined_df.sort_values('composite_score', ascending=False)
    
    output_filename = "combined_winger_analysis.csv"
    combined_df.to_csv(output_filename, index=False)
    print(f"Analysis complete. Found {len(combined_df)} unique wingers.")
    print(f"Results saved to {output_filename}")
    
    return combined_df

if __name__ == "__main__":
    leagues = [
        {"competition_id": 9, "season_ids": [281]},
        {"competition_id": 43, "season_ids": [106]},
        {"competition_id": 11, "season_ids": [90, 42, 4]},
        {"competition_id": 7, "season_ids": [235, 108]},
        {"competition_id": 2, "season_ids": [44]},
        {"competition_id": 12, "season_ids": [27]},
        {"competition_id": 55, "season_ids": [282]},
    ]
    
    result_df = analyze_multiple_leagues(leagues)
    
    if not result_df.empty:
        print("\nTop 10 Wingers Across All Leagues:")
        print("-" * 100)
        display_columns = [
            'player', 'team', 'competition_id', 'season_id',
            'matches_played', 'minutes_played',
            'finishing_quality', 'chance_creation', 'ball_progression',
            'one_v_one', 'off_ball_movement', 'defensive_contribution',
            'composite_score'
        ]
        print(result_df[display_columns].head(10))